In [1]:
#Subquery ek query ke andar  dusri query hoti hai. Three  main types hote hai:

In [2]:
#1.Scaler subquery - ek single value return karti hai.
#2.Row/multi-row subquery - IN, NOT IN ke saath multiple values.
#Correlated subquery - outer query ki har row ke liye alag se chalta hai(thoda slow, lekin powerful)

In [3]:
#01:Scalar Subquery - WHERE clause me single value compare karna:

In [4]:
import pandas as pd
import sqlite3

conn = sqlite3.connect('../data/db/ecommerce.db')


# un customers ko nikalo jinka total spend overall average se jyada hai
q1 = pd.read_sql("""
    SELECT "Customer ID", SUM(OrderValue) as total_spend
    FROM orders
    GROUP BY "Customer ID"
    HAVING SUM(OrderValue) > (
        SELECT AVG(total_spend) FROM (
            SELECT SUM(OrderValue) as total_spend FROM orders GROUP BY "Customer ID"
        )
    )
    ORDER BY total_spend DESC
    LIMIT 10
""",conn)
print(q1)

   Customer ID  total_spend
0      18102.0    598215.22
1      14646.0    523342.07
2      14156.0    296564.69
3      14911.0    270248.53
4      17450.0    233579.39
5      13694.0    190825.52
6      17511.0    171885.98
7      12415.0    143269.29
8      16684.0    141502.25
9      15061.0    136391.48


In [5]:
#02:IN subquery - multi-row subquery:

In [6]:
# Un customers ke details nikaalo jo Germany ke hain (customers table se), aur unke orders bhi dikhao (orders table se)
q2 = pd.read_sql("""
    SELECT *
    FROM orders
    WHERE "Customer ID" IN (
        SELECT "Customer ID" FROM customers WHERE Country = 'Germany'
    )
    ORDER BY OrderValue DESC
    LIMIT 10
""", conn)
print(q2)

  Invoice  Customer ID          InvoiceDate  OrderValue
0  552978      12590.0  2011-05-12 14:46:00    9341.260
1  564856      12477.0  2011-08-31 09:11:00    4257.060
2  530799      12497.0  2010-11-04 12:31:00    3993.640
3  505335      12709.0  2010-04-21 13:00:00    3857.200
4  537201      12472.0  2010-12-05 14:19:00    3262.600
5  504146      12481.0  2010-04-11 13:10:00    2969.600
6  501545      12709.0  2010-03-17 14:59:00    2917.760
7  516131      12709.0  2010-07-16 15:26:00    2903.710
8  504332      12671.0  2010-04-12 16:30:00    2622.481
9  530970      12477.0  2010-11-05 09:35:00    2570.770


In [7]:
#03:NOT IN — un customers ko dhoondo jinka koi order nahi hai (JOIN ke alawa subquery se bhi same kaam hota hai):

In [8]:
q3 = pd.read_sql("""
    SELECT "Customer ID", Country
    FROM customers
    WHERE "Customer ID" NOT IN (
        SELECT "Customer ID" FROM orders WHERE "Customer ID" IS NOT NULL
    )
""", conn)
print(q3.shape)
print(q3)

(0, 2)
Empty DataFrame
Columns: [Customer ID, Country]
Index: []


In [9]:
#04:Correlated Subquery — outer query ki har row ke liye subquery re-run hota hai:

In [10]:
# Har customer ka sabse bada single order dikhao (uske apne orders mein se)
q4 = pd.read_sql("""
    SELECT o1."Customer ID", o1.Invoice, o1.OrderValue
    FROM orders o1
    WHERE o1.OrderValue = (
        SELECT MAX(o2.OrderValue)
        FROM orders o2
        WHERE o2."Customer ID" = o1."Customer ID"
    )
    ORDER BY o1.OrderValue DESC
    LIMIT 10
""", conn)
print(q4)

   Customer ID Invoice  OrderValue
0      16446.0  581483   168469.60
1      12346.0  541431    77183.60
2      14156.0  493819    44051.60
3      15098.0  556444    38970.00
4      17450.0  524181    33167.80
5      18102.0  537659    31770.98
6      12415.0  556917    22775.93
7      15749.0  550461    21535.90
8      14646.0  572035    20277.92
9      12931.0  562439    18841.48


In [12]:
# Yahan inner query (o2) outer query (o1) ke "Customer ID" ko reference kar rahi hai — isiliye ise "correlated" kehte hain, yeh har outer row ke liye alag se evaluate hoti hai.

In [13]:
#05:Subquery in FROM clause (derived table) — already Day 12-14 mein use kar chuke ho, ek aur example:

In [14]:
# Sirf un countries ka data dikhao jinka total revenue 10000 se zyada hai
q5 = pd.read_sql("""
    SELECT *
    FROM (
        SELECT c.Country, SUM(o.OrderValue) as total_revenue
        FROM customers c
        INNER JOIN orders o ON c."Customer ID" = o."Customer ID"
        GROUP BY c.Country
    )
    WHERE total_revenue > 10000
    ORDER BY total_revenue DESC
""", conn)
print(q5)

            Country  total_revenue
0    United Kingdom   1.380662e+07
1              EIRE   5.792255e+05
2       Netherlands   5.485249e+05
3           Germany   4.189031e+05
4            France   3.268431e+05
5         Australia   1.722312e+05
6             Spain   1.037355e+05
7       Switzerland   1.010288e+05
8            Sweden   8.745542e+04
9           Belgium   7.415863e+04
10          Denmark   7.294083e+04
11         Portugal   5.240545e+04
12            Japan   4.377658e+04
13  Channel Islands   4.145361e+04
14           Norway   3.924303e+04
15          Austria   3.344362e+04
16            Italy   3.067935e+04
17          Finland   2.951445e+04
18           Cyprus   2.825848e+04
19           Greece   1.899549e+04
20        Singapore   1.315816e+04
21           Poland   1.052809e+04
22           Israel   1.019365e+04


practice questions.

In [16]:
#1.EXISTS subquery try karo — un customers ko dhoondo jinka kam se kam ek order OrderValue > 1000 ka hai (WHERE EXISTS (SELECT 1 FROM orders WHERE ...)).

In [17]:
#

In [18]:
#2.Correlated subquery se har country ka "second highest spending customer" nikaalne ki koshish karo (thoda challenging hai — Window Function se aasan hota, lekin subquery se try karna practice ke liye achha hai).

In [19]:
#

In [20]:
conn.close()